### Imports

In [ ]:
import json
import re, os
from pathlib import Path
os.getcwd()
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)

### File path

In [ ]:
DATA_PATH = Path("accounts-bills.json")
assert DATA_PATH.exists(), f"{DATA_PATH} not found in {Path.cwd()}"

with open(DATA_PATH, encoding="utf-8") as f:
    raw = json.load(f)

df = pd.json_normalize(raw)

print(f"Columns: {df.shape[1]}, Rows: {df.shape[0]}")
df.head()

### Main Columns

In [ ]:
EXPECTED_COLUMNS = [
    "vendorId", "itemName", "accountName", "itemTotalAmount"
]

missing_cols = [c for c in EXPECTED_COLUMNS if c not in df.columns]
assert not missing_cols, f"Missing columns: {missing_cols}"

df.info()

### Check Null

In [ ]:
df_check = df.replace("", np.nan)

missing = df_check.isnull().sum()
missing_pct = (missing / len(df)) * 100

missing_df = pd.DataFrame({
    "missing_count": missing,
    "missing_pct": missing_pct
}).sort_values("missing_count", ascending=False)

missing_df[missing_df["missing_count"] > 0]

In [ ]:
missing_df["missing_count"].plot(kind="barh", figsize=(8, 5))
plt.title("Missing Values per Column")
plt.xlabel("Count")
plt.show()

In [ ]:
vc = df["accountName"].value_counts()

print("Total classes:", len(vc))
print("Imbalance ratio:", vc.max() / vc.min())
print("Classes <5 samples:", (vc < 5).sum())

vc.head(20).sort_values().plot(kind="barh", figsize=(10, 6))
plt.title("Top 20 Categories")
plt.show()

### Amount

In [ ]:
amounts = df["itemTotalAmount"]

print(amounts.describe())

plt.hist(amounts, bins=50)
plt.title("Raw Amount Distribution")
plt.show()

plt.hist(np.log1p(np.abs(amounts)), bins=50)
plt.title("Log Amount Distribution")
plt.show()

### Vendor grouping

In [ ]:
vendor_map = (
    df.groupby("vendorId")["accountName"]
      .agg(lambda x: x.value_counts().index[0])
)

df["vendor_pred"] = df["vendorId"].map(vendor_map)

vendor_acc = (df["vendor_pred"] == df["accountName"]).mean()

print("Vendor-only accuracy:", vendor_acc)

### Vendor + Item Name

In [ ]:
df["clean_name"] = df["itemName"].str.lower().str.strip()

df["vkey"] = df["vendorId"] + "|" + df["clean_name"]

key_map = (
    df.groupby("vkey")["accountName"]
      .agg(lambda x: x.value_counts().index[0])
)

df["key_pred"] = df["vkey"].map(key_map)

key_acc = (df["key_pred"] == df["accountName"]).mean()

print("Vendor + itemName accuracy:", key_acc)

In [ ]:
print(f"Records: {len(df)}")
print(f"Classes: {df['accountName'].nunique()}")

print(f"\nVendor signal: {vendor_acc:.4f}")
print(f"Vendor + name signal: {key_acc:.4f}")

print("\nTop 5 categories:")
print(df["accountName"].value_counts().head())